# 05｜Poisson 計數模型
沿用第 04 冊的時間切分與 5 天尖峰門檻。以歷史日型平均為起點，用上一小時借車量修正預測。

In [1]:
from zipfile import ZipFile
import pandas as pd
start, split, end = map(pd.Timestamp, ['2025-01-01', '2025-01-18', '2025-01-25'])
with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, usecols=['started_at', 'start_station_id'], dtype='string')
hour = pd.to_datetime(rides.started_at, format='mixed', errors='coerce').dt.floor('h')
station = rides.start_station_id.str.strip().replace('', pd.NA)
events = pd.DataFrame({'station': station, 'hour': hour})
events = events.loc[hour.ge(start) & hour.lt(end) & station.notna()]
stations = sorted(events.loc[events.hour.lt(split) & ~events.hour.dt.day.isin([1, 9]), 'station'].unique())
cohort = events[events.station.isin(stations)]
grid = pd.MultiIndex.from_product(
    [stations, pd.date_range(start, end, freq='h', inclusive='left')],
    names=['station', 'hour'])
panel = cohort.groupby(['station', 'hour']).size().reindex(grid, fill_value=0).rename('actual').to_frame()
assert panel.actual.sum() == len(cohort)


In [2]:
p = panel.reset_index()
p['lag'] = p.groupby('station').actual.shift(1)
p['date'] = p.hour.dt.normalize()
p['h'] = p.hour.dt.hour
p['weekend'] = p.hour.dt.dayofweek >= 5
ordinary = p.loc[~p.hour.dt.day.isin([1, 9, 20])]
train = ordinary.loc[ordinary.hour < split]
valid = ordinary.loc[ordinary.hour >= split].copy()
keys = ['station', 'weekend', 'h']
valid = valid.join(train.groupby(keys).actual.mean().rename('history'), on=keys)
assert valid[['history', 'lag']].notna().all().all()
weekday = train.loc[~train.weekend]

import numpy as np
# Leave the target date out of its training historical-average feature.
g = train.groupby(keys).actual
history_loo = (g.transform('sum') - train.actual) / (g.transform('count') - 1)
fit = train.assign(history=history_loo).dropna(subset=['history','lag'])
def design(df):
    return np.column_stack([np.ones(len(df)), np.log1p(df.history), np.log1p(df.lag)])
X, y = design(fit), fit.actual.to_numpy()
beta = np.zeros(X.shape[1])
# Newton updates for the Poisson log-link likelihood.
for _ in range(40):
    mu = np.exp(X @ beta)
    step = np.linalg.solve(X.T @ (mu[:, None] * X), X.T @ (y - mu))
    beta += step
    if np.max(np.abs(step)) < 1e-8:
        break
else:
    raise RuntimeError('Poisson fit did not converge')
valid['poisson'] = np.exp(design(valid) @ beta)
assert np.isfinite(valid.poisson).all()
def score(label, frame):
    row = [label, frame.station.nunique(), len(frame)]
    for pred in ['lag', 'history', 'poisson']:
        gap = frame.actual - frame[pred]
        row.extend([gap.abs().mean(), gap.clip(lower=0).mean()])
    return row
rows = [score('all', valid)]
for label, lo, hi in [('AM', 6, 10), ('PM', 15, 20)]:
    window = weekday.loc[weekday.h.between(lo, hi-1)]
    days = window.loc[window.actual.gt(0)].groupby('station').date.nunique()
    hourly = window.groupby(['station','h']).actual.mean().unstack()
    pairs = hourly.iloc[:, :-1] + hourly.shift(-1, axis=1).iloc[:, :-1]
    peak = pairs.idxmax(axis=1)
    for threshold in (5,):
        eligible = days[days.ge(threshold)].index
        v = valid.loc[~valid.weekend & valid.station.isin(eligible)]
        first = v.station.map(peak)
        rows.append(score(f'{label}/{threshold}', v.loc[(v.h == first) | (v.h == first + 1)]))
report = pd.DataFrame(rows, columns=['scope','stations','n','lag_MAE','lag_under','hist_MAE','hist_under','model_MAE','model_under'])
print(report.round(4).to_string(index=False))

scope  stations      n  lag_MAE  lag_under  hist_MAE  hist_under  model_MAE  model_under
  all       748 107712   0.3663     0.1831    0.3066      0.1661     0.3661       0.1921
 AM/5       357   2856   1.1541     0.7906    0.9046      0.5401     0.9915       0.6679
 PM/5       453   3624   1.1598     0.6912    0.9414      0.5006     1.0789       0.5815


訓練歷史平均排除該列自身，驗證只用訓練期平均。留一日法在此用於建立訓練特徵，同一站日型小時每天只有一列。這是探索性比較，需以新月份複驗。